# **Data warehouse: MobilityDB** (native `tgeompoint` + GiST index)

The warehouse table is just identity + attributes + the trajectory, and the index prunes on
the space-*time* box. Queries rely solely on `traj && stbox(env, tspan)`; no `dt` filter.

Reported per run: **query time + index candidate-set size (selectivity) + storage (heap+index)**.

In [ ]:
import os, glob, time, json, statistics
from datetime import datetime, timedelta
import psycopg2, pandas as pd
PGCONN="dbname=mdb_warehouse"    
TZ="Europe/Copenhagen"; CLI=os.getcwd()
L0DIR=os.path.join(CLI,"data/trips/L0/L0")
GLOB=f"{L0DIR}/year=2026/month=01/day=*.parquet"; FILES=sorted(glob.glob(GLOB)); print(len(FILES),"L0 files")
T0,T1,TMID="2026-01-15 08:00:00","2026-01-16 08:00:00","2026-01-15 20:00:00"
BELT="640730.0, 6042487.0, 654100.0, 6058230.0"
N_ITERS,TRIM,WARMUP=5,1,1

31 L0 files


## 1. Load the L0 segments natively 

In [2]:
pg=psycopg2.connect(PGCONN); pg.autocommit=True; cur=pg.cursor()
cur.execute(f"SET TimeZone='{TZ}';")
for ext in ("postgis","mobilitydb CASCADE","pg_parquet"): cur.execute(f"CREATE EXTENSION IF NOT EXISTS {ext};")
cur.execute("DROP TABLE IF EXISTS l0_raw, l0_wh;")
# raw load (parquet carries the sidecars; we discard them in the next step)
cur.execute("""CREATE TABLE l0_raw(mmsi bigint, ship_type text, segment_type text,
   traj bytea, tmin timestamptz, tmax timestamptz,
   bxmin float8, bxmax float8, ymin float8, ymax float8, dt date);""")
t=time.time()
for f in FILES: cur.execute(f"COPY l0_raw FROM '{f}' (FORMAT parquet);")
load_raw_s=time.time()-t
# native warehouse table: identity + attributes + trajectory. NO sidecar stats, NO dt.
t=time.time()
cur.execute("""CREATE TABLE l0_wh AS
   SELECT mmsi, ship_type, segment_type, tgeompointFromEWKB(traj) AS traj FROM l0_raw;""")
materialize_s=time.time()-t
cur.execute("DROP TABLE l0_raw;")
cur.execute("SELECT count(*) FROM l0_wh;"); n=cur.fetchone()[0]
print(f"loaded {n:,} segments natively (traj only)  raw copy {load_raw_s:.1f}s + decode {materialize_s:.1f}s")

loaded 381,713 segments natively (traj only)  raw copy 144.7s + decode 300.0s


## 2. Build the native GiST spatiotemporal index (the warehouse's only pruning structure) + size

In [3]:
t=time.time()
cur.execute("CREATE INDEX l0_wh_gist ON l0_wh USING gist(traj);")
cur.execute("ANALYZE l0_wh;")
index_build_s=time.time()-t
cur.execute("SELECT pg_size_pretty(pg_relation_size('l0_wh')),"
            " pg_size_pretty(pg_relation_size('l0_wh_gist')),"
            " pg_size_pretty(pg_total_relation_size('l0_wh'));")
heap,idx,total=cur.fetchone()
print(f"GiST built in {index_build_s:.1f}s   heap={heap}  index={idx}  total={total}")
print("compare 'total' with the Parquet layer (~3.3-3.9 GB, open, no server) - the index has no lakehouse counterpart")

GiST built in 21.3s   heap=219 MB  index=75 MB  total=5916 MB
compare 'total' with your Parquet layer (~3.3-3.9 GB, open, no server) - the index has no lakehouse counterpart


## 3. Queries: native, index-only pruning (`traj && stbox`)

In [ ]:
SIDECAR_SHIFT = timedelta(hours=1)
_F = "%Y-%m-%d %H:%M:%S"
def _shift(ts): return (datetime.strptime(ts,_F)-SIDECAR_SHIFT).strftime(_F)
P0,P1 = _shift(T0), _shift(T1)

def sbox(env,t0=T0,t1=T1): return f"stbox(ST_MakeEnvelope({env}), tstzspan('[{t0}, {t1})'))"
def pbox(env): return sbox(env,P0,P1)          # prune box, sidecar clock
BELT_SB=sbox(BELT); BELT_PB=pbox(BELT)
CLIPPED=f"""WITH clipped AS (
   SELECT mmsi, ship_type, atStbox(traj, {BELT_SB}) g FROM l0_wh
   WHERE traj && {BELT_PB}),
 c AS (SELECT mmsi, ship_type, g FROM clipped WHERE g IS NOT NULL)"""
def prox(margin,agg):
    return f"""{CLIPPED},
   ext AS (SELECT mmsi,g,startTimestamp(g) ts0,endTimestamp(g) ts1,
                  ST_XMin(e) x0,ST_XMax(e) x1,ST_YMin(e) y0,ST_YMax(e) y1
           FROM (SELECT mmsi,g,ST_Envelope(trajectory(g)) e FROM c) s WHERE e IS NOT NULL),
   cand2 AS (SELECT a.mmsi m1,b.mmsi m2,a.g t1,b.g t2 FROM ext a JOIN ext b
             ON a.mmsi<b.mmsi AND a.ts0<b.ts1 AND a.ts1>b.ts0
             AND a.x0<=b.x1+{margin} AND b.x0<=a.x1+{margin}
             AND a.y0<=b.y1+{margin} AND b.y0<=a.y1+{margin})
   {agg}"""
QUERIES={
 "clip_to_region": f"{CLIPPED} SELECT count(DISTINCT mmsi) FROM c;",
 "harbour_entry": f"""SELECT count(DISTINCT mmsi) FROM (
     SELECT mmsi, atTime(traj, tstzspan('[{T0}, {T1})')) trip FROM l0_wh
     WHERE traj && {pbox('666538.0,6392057.0,679171.0,6403745.0')}) s
   WHERE trip IS NOT NULL AND eIntersects(trip, ST_MakeEnvelope(666538.0,6392057.0,679171.0,6403745.0));""",
 "both_ports": f"""WITH rodby AS (SELECT DISTINCT mmsi FROM l0_wh
       WHERE traj && {pbox('651135.0,6058230.0,651422.0,6058548.0')}
         AND eIntersects(traj, ST_MakeEnvelope(651135.0,6058230.0,651422.0,6058548.0))),
      putt AS (SELECT DISTINCT mmsi FROM l0_wh
       WHERE traj && {pbox('644339.0,6042108.0,644896.0,6042487.0')}
         AND eIntersects(traj, ST_MakeEnvelope(644339.0,6042108.0,644896.0,6042487.0)))
   SELECT count(*) FROM rodby JOIN putt USING(mmsi);""",
 "position_interpolation": f"""{CLIPPED}
   SELECT count(DISTINCT mmsi) FROM (SELECT mmsi, valueAtTimestamp(g, TIMESTAMP '{TMID}') p FROM c) s WHERE p IS NOT NULL;""",
 "collision":      prox(300,"SELECT count(*) FROM (SELECT DISTINCT m1,m2 FROM cand2 WHERE nearestApproachDistance(t1,t2)<300) s;"),
 "encounter_zone": prox(500,"SELECT count(*) FROM (SELECT DISTINCT m1,m2 FROM cand2 WHERE nearestApproachDistance(t1,t2)<500) s;"),
 "nearest_approach":prox(2000,"SELECT round(min(nearestApproachDistance(t1,t2))) FROM cand2;"),
 "fleet_summary":  f"{CLIPPED} SELECT round((sum(length(g))/1000.0)::numeric,1) FROM c;",
 "bounding_box":   f"""{CLIPPED}
   SELECT round(avg((x1-x0)*(y1-y0)/1e6)::numeric,2) FROM (
     SELECT mmsi, min(x0) x0, max(x1) x1, min(y0) y0, max(y1) y1 FROM (
       SELECT mmsi, ST_XMin(trajectory(g)) x0, ST_XMax(trajectory(g)) x1,
              ST_YMin(trajectory(g)) y0, ST_YMax(trajectory(g)) y1 FROM c) s GROUP BY mmsi) t;""",
 "speed_profile":  f"""{CLIPPED}
   SELECT round((percentile_cont(0.5) WITHIN GROUP (ORDER BY vmax)*1.94384)::numeric,1) FROM (
     SELECT mmsi, max(maxValue(speed(g))) vmax FROM c GROUP BY mmsi) s;""",
}
ORDER=["clip_to_region","harbour_entry","both_ports","position_interpolation",
       "collision","encounter_zone","nearest_approach","fleet_summary","bounding_box","speed_profile"]
print("EXPLAIN one - confirm 'Index Scan using l0_wh_gist', not Seq Scan:")
cur.execute("EXPLAIN (ANALYZE, BUFFERS) "+QUERIES["clip_to_region"])
print("\n".join(r[0] for r in cur.fetchall()[:8]))

EXPLAIN one - confirm 'Index Scan using l0_wh_gist', not Seq Scan:


Aggregate  (cost=641.73..641.74 rows=1 width=8) (actual time=297.378..297.379 rows=1 loops=1)
  Buffers: shared hit=6673 read=6035
  ->  Sort  (cost=640.92..641.33 rows=161 width=8) (actual time=297.353..297.362 rows=440 loops=1)
        Sort Key: l0_wh.mmsi
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=6673 read=6035
        ->  Bitmap Heap Scan on l0_wh  (cost=21.54..635.02 rows=161 width=8) (actual time=23.275..297.311 rows=440 loops=1)
              Recheck Cond: (traj && 'STBOX XT(((640730,6042487),(654100,6058230)),[2026-01-15 07:00:00+01, 2026-01-16 07:00:00+01))'::stbox)


## 4. Runtime + **selectivity** (index candidate rows) + answer, per query

In [5]:
def trimmed(xs):
    xs=sorted(xs); xs=xs[TRIM:len(xs)-TRIM] if len(xs)>2*TRIM else xs
    return statistics.mean(xs)

def index_candidate_rows(sql):
    """Sum 'Actual Rows' of every GiST Index Scan node on l0_wh_gist = the rows the index
    hands to the exact MEOS predicate (the warehouse's selectivity, engine-independent)."""
    cur.execute("EXPLAIN (ANALYZE, FORMAT JSON) "+sql)
    plan=cur.fetchone()[0]
    if isinstance(plan,str): plan=json.loads(plan)
    total=[0]
    def walk(node):
        if node.get("Index Name")=="l0_wh_gist":
            total[0]+=node.get("Actual Rows",0)*node.get("Actual Loops",1)
        for k in ("Plans","Plan"):
            ch=node.get(k)
            if isinstance(ch,list):
                for c in ch: walk(c)
            elif isinstance(ch,dict): walk(ch)
    walk(plan[0]["Plan"] if isinstance(plan,list) else plan["Plan"])
    return total[0]

rows=[]
for q in ORDER:
    sql=QUERIES[q]
    for _ in range(WARMUP): cur.execute(sql); cur.fetchall()
    ts=[]; ans=None
    for _ in range(N_ITERS):
        t=time.time(); cur.execute(sql); ans=cur.fetchone()[0]; ts.append(time.time()-t)
    try: cand=index_candidate_rows(sql)
    except Exception as e: cand=f"ERR:{type(e).__name__}"
    rows.append({"query":q,"answer":ans,"warehouse_mdb_ms":round(1000*trimmed(ts),1),
                 "index_candidate_rows":cand})
    print(f"{q:24} answer={str(ans):>10}  {1000*trimmed(ts):9.1f} ms   candidates={cand}")
df=pd.DataFrame(rows); df.to_csv("results/mobilitydb_warehouse.csv",index=False); df

clip_to_region           answer=       115      102.4 ms   candidates=541


harbour_entry            answer=        99      188.5 ms   candidates=1139


both_ports               answer=         4      256.0 ms   candidates=174


position_interpolation   answer=        29      148.6 ms   candidates=541


collision                answer=       163      862.2 ms   candidates=541


...

nearest_approach         answer=       0.0      901.2 ms   candidates=541


fleet_summary            answer=    2840.5      146.1 ms   candidates=541


bounding_box             answer=     62.69      300.4 ms   candidates=541


speed_profile            answer=      35.9      155.0 ms   candidates=541


,query,answer,warehouse_mdb_ms,index_candidate_rows
0,clip_to_region,115,102.4,541
1,harbour_entry,99,188.5,1139
2,both_ports,4,256.0,174
3,position_interpolation,29,148.6,541
4,collision,163,862.2,541
5,encounter_zone,194,872.5,541
6,nearest_approach,0.0,901.2,541
7,fleet_summary,2840.5,146.1,541
8,bounding_box,62.69,300.4,541
9,speed_profile,35.9,155.0,541
